# Mini-expérience : A0 à A5 sur NQ et TriviaQA

`MODE` choisit le run : `essai` (A1, NQ, 10 questions de mise au point), `dev` (A0 à A3, 10 questions de mise au point
par jeu), `complet` (les 12 runs, 100 questions, graine 42). `ONLY` relance seulement certains couples.

Un processus par run, ce qui libère la mémoire du GPU. Un run qui échoue n'arrête pas les suivants ; la dernière cellule
lève une erreur s'il y en a eu un. Sorties : `results/`, `logs/`, `pip_freeze.txt`.

In [ ]:
MODE = "dev"
ONLY = [("A2", "nq"), ("A2", "triviaqa"), ("A3", "nq"), ("A3", "triviaqa")]  # ou None
BRANCH = "preparation-volet-a"


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working /tmp | cat
import subprocess
from pathlib import Path

REPO = "vqlentinn/memoire-rl-search"
CLONE = Path("/tmp/memoire-rl-search")  # hors de /kaggle/working : seule la sortie utile est publiée
if not CLONE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(CLONE)], check=True)
print(subprocess.run(["git", "-C", str(CLONE), "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q vllm==0.33.0 bm25s PyStemmer datasets huggingface_hub faiss-cpu 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze.txt
!python -c "import vllm, torch, faiss; print('vllm', vllm.__version__, '| torch', torch.__version__, '| gpus', torch.cuda.device_count(), '| faiss', faiss.__version__)"


In [ ]:
import os, shutil, time

QWEN = "Qwen/Qwen2.5-3B-Instruct"
A4 = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-em-ppo-v0.3"
A5 = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-3b-em-grpo-v0.3"
MODELS = {"A0": QWEN, "A1": QWEN, "A2": QWEN, "A3": QWEN, "A4": A4, "A5": A5}
DATASETS = ["nq", "triviaqa"]

if MODE == "essai":
    jobs, n, seed = [("A1", "nq")], 10, 7
elif MODE == "dev":
    jobs, n, seed = [(s, d) for s in ["A0", "A1", "A2", "A3"] for d in DATASETS], 10, 7
elif MODE == "complet":
    jobs, n, seed = [(s, d) for s in MODELS for d in DATASETS], 100, 42
else:
    raise ValueError(MODE)
if ONLY:
    jobs = [j for j in jobs if j in ONLY]

WORK = Path("/kaggle/working")
outdir = WORK / ("results" if MODE == "complet" else f"results_{MODE}")
(WORK / "logs").mkdir(exist_ok=True)
env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn")
hub = Path.home() / ".cache/huggingface/hub"

status = []
for i, (system, dataset) in enumerate(jobs):
    cmd = ["python", "-u", "scripts/run_volet_a.py", "--system", system, "--dataset", dataset, "--n", str(n),
           "--seed", str(seed), "--corpus", "1m", "--retriever", "e5", "--model", MODELS[system], "--outdir", str(outdir)]
    log = WORK / "logs" / f"{MODE}_{system}_{dataset}.log"
    print("===", " ".join(cmd), flush=True)
    t0 = time.time()
    with log.open("a") as f:
        p = subprocess.Popen(cmd, cwd=CLONE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line); f.flush()
            print(line, end="", flush=True)
        rc = p.wait()
    status.append((system, dataset, rc, round(time.time() - t0)))
    print(f"=== {system} {dataset} : code {rc}, {status[-1][3]} s", flush=True)
    # Trois modèles de 6 à 14 Go : on efface celui dont aucun run restant n'a besoin.
    if MODELS[system] not in {MODELS[s] for s, _ in jobs[i + 1:]}:
        shutil.rmtree(hub / ("models--" + MODELS[system].replace("/", "--")), ignore_errors=True)


In [ ]:
for row in status:
    print(*row)
echecs = [s for s in status if s[2] != 0]
if echecs:
    raise RuntimeError(f"Runs en échec : {echecs}")
print("Tous les runs ont rendu le code 0 : vérifier quand même les journaux et les trajectoires.")
